# Initialize

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType,DateType
from pyspark.sql.functions import trim,col
from pyspark.sql.window import Window

# Read Bronze table "crm_sales_details"

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

# Silver Transformations

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## CID cleanup


In [0]:
df = df.withColumn("cid",F.regexp_replace(col("cid"),"-",""))

## Cntry Normalization

In [0]:

df = df.withColumn(
    "cntry",
    F.when(F.col("cntry").isin("USA", "US"), "United States")
    .when(col("cntry") == "DE","Germany")
    .when(col("cntry").isNull() | (col("cntry") == ''), "n/a")
    .otherwise(col("cntry"))
)
display(df)    

# Renaming Columns

In [0]:
RENAME_COLLUMNS ={
    "cid":"customer_number",
    "cntry":"country",
}
for old_name, new_name in RENAME_COLLUMNS.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity checks of Dataframe

In [0]:
df.limit(10).display()

# Write Silver Table

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customer_location')

## Sanity Checks for Silver Table

In [0]:
%sql 

SELECT * FROM workspace.silver.erp_customer_location LIMIT(10)